# Kepler threshold-crossing bias (Track B Stage 3)

Figures for [docs/kepler_threshold_bias_paper.md](../docs/kepler_threshold_bias_paper.md).
Loads the aggregates produced by
`scripts/validate/analyze_kepler_threshold_bias.py` and plots the three
headline relations. Run the script first:

```bash
docker compose run --rm pipeline python -m scripts.validate.analyze_kepler_threshold_bias \
    --out-prefix data/output/kepler_bias/threshold
```

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

PREFIX = Path('../data/output/kepler_bias/threshold')
summary = json.loads(PREFIX.with_name('threshold_summary.json').read_text())
bands = pl.read_csv(PREFIX.with_name('threshold_bands.csv'))
head = summary['headline']
print(f"refined pairs : {head['n_refined']:,.0f}")
print(f"cross up      : {head['n_cross_up']:,.0f}  ({head['cross_up_rate_of_kepler_below']:.3%} of Kepler<0.05)")
print(f"cross down    : {head['n_cross_down']:,.0f}  (censored: catalog has no Kepler>=0.05 pairs)")
print(f"delta_dist    : mean={head['mean_delta']:.2e}  median={head['median_delta']:.2e}  std={head['std_delta']:.2e} AU")

## 1. Crossing rate vs proximity to the threshold

Essentially all crossings live in the last 0.005 AU band below 0.05 — the
over-detection is an edge effect of the distance cut.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = bands['kepler_band_lo_au'].to_numpy()
ax.bar(x, bands['cross_up_rate'].to_numpy() * 100, width=0.0045, align='edge', color='C3', alpha=0.8)
ax.set_xlabel('Kepler min-distance band (AU)')
ax.set_ylabel('upward crossing rate (%)')
ax.set_title('Kepler->N-body crossing rate vs distance to threshold (0.05 AU)')
ax.axvline(0.05, ls='--', c='k', lw=1)
fig.tight_layout()

## 2. Mean correction per band

Delta = d_Nbody - d_Kepler is slightly **negative** on average (N-body tends
to tighten), against a std ~4e-4 AU. The crossings are driven by scatter, not
by a one-directional shift.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(bands['kepler_band_lo_au'].to_numpy() + 0.0025, bands['mean_delta_au'].to_numpy() * 1e6, 'o-')
ax.axhline(0, ls='--', c='k', lw=1)
ax.set_xlabel('Kepler min-distance band centre (AU)')
ax.set_ylabel('mean (d_Nbody - d_Kepler)  (micro-AU)')
ax.set_title('Mean N-body correction per band')
fig.tight_layout()

## 3. Velocity dependence and orbital band of crossings

In [ ]:
vq = pl.DataFrame(summary['rel_vel_quintiles'])
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(vq['quintile'].to_numpy(), vq['cross_up_rate'].to_numpy() * 100, 's-')
ax.set_xlabel('relative-velocity quintile (1=slow, 5=fast)')
ax.set_ylabel('upward crossing rate (%)')
ax.set_title('Faster encounters cross more often')
fig.tight_layout()

orb = summary['orbital_band_of_crossings']
print(f"crossings with q_min<1.8 AU : {orb['n_q_min_lt_1p8']:,.0f}/{orb['n_joined']:,.0f} ({orb['n_q_min_lt_1p8']/orb['n_joined']:.1%})")
print(f"crossings with e_max>0.3    : {orb['n_e_max_gt_0p3']:,.0f}/{orb['n_joined']:,.0f} ({orb['n_e_max_gt_0p3']/orb['n_joined']:.1%})")
print(f"median q_min={orb['median_q_min']:.2f} AU, median e_max={orb['median_e_max']:.3f}")

**Takeaway.** The 25,283 upward crossings (0.29% of refined Kepler<0.05 pairs)
are an edge-plus-scatter effect concentrated at 0.045-0.050 AU, in low-perihelion
/ high-eccentricity / fast-encounter orbits. The 'zero downward crossings' is
censoring (the catalog excludes Kepler>=0.05 pairs), so the false-negative rate
is *not* zero and is unmeasurable here. See the draft note for the experiment
that would measure it.